# 🔒 Security & Privacy — UC Column Masking

Attaches a Unity Catalog mask function to the real PII columns on Silver
(confirmed via README §6.1–6.2): `candidate_name`, `application_id`,
`gender`. Every query against a masked column runs the mask function
first — non-admins see `***`, `admins` group members see the real value.

**Scope:** Silver only. Gold (`gold.mhcet_cutoffs`, `gold.mhcet_cutoffs_by_pool`)
is pre-aggregated and never selects these columns, so it's PII-free by
construction and untouched here. The app queries Gold, so it is unaffected.

**Feasibility check (done ahead of time via the SQL warehouse against the
`sandbox` catalog):** both `is_account_group_member('admins')` and the
workspace-local `is_member('admins')` resolve without error in this
workspace — account-level UC groups are available, so we use
`is_account_group_member` (the UC-recommended one, consistent across any
workspace attached to this metastore). Also confirmed there: the mask
survives a full table rewrite, so this is a one-time setup step, not part
of the recurring Bronze→Silver→Gold→DQ chain — see `PIPELINE_ORDER.md`.

**Output:** masking function + `SET MASK` on `{CATALOG}.silver.mhcet_allotments`.

**Before running on Databricks:** this file targets the real catalog
(`rankrangers_project_data`) by design — it should not be committed with a
`sandbox` override. To test safely first, edit only the *workspace copy*
you push to Databricks (change Cell 1's `CATALOG` to `'sandbox'` there),
verify masking behavior, then re-push this unmodified file to run against
production.

## Cell 1 — Config

In [ ]:
# NOTE: test this notebook against CATALOG = 'sandbox' in the Databricks
# workspace copy before running it against the real catalog below - do not
# commit a 'sandbox' override to this file (see PIPELINE_ORDER.md).
CATALOG    = 'rankrangers_project_data'
SCHEMA     = 'silver'
TABLE      = 'mhcet_allotments'
FULL_TABLE = f'{CATALOG}.{SCHEMA}.{TABLE}'
MASK_FUNC  = f'{CATALOG}.{SCHEMA}.mask_pii'

PII_COLUMNS = ['candidate_name', 'application_id', 'gender']

print(f'Target table   : {FULL_TABLE}')
print(f'Mask function  : {MASK_FUNC}')
print(f'Columns to mask: {PII_COLUMNS}')

## Cell 2 — Create Masking Function

Members of the `admins` account group see the real value; everyone else
sees `***`.

In [ ]:
spark.sql(f'''
CREATE OR REPLACE FUNCTION {MASK_FUNC}(val STRING)
RETURNS STRING
RETURN CASE WHEN is_account_group_member('admins') THEN val ELSE '***' END
''')

print(f'✅ Created {MASK_FUNC}')

## Cell 3 — Attach Mask to PII Columns

In [ ]:
for col in PII_COLUMNS:
    spark.sql(f'ALTER TABLE {FULL_TABLE} ALTER COLUMN {col} SET MASK {MASK_FUNC}')
    print(f'✅ Mask attached: {FULL_TABLE}.{col}')

## Cell 4 — Verify Masking (as yourself, non-admin)

You are not currently a member of the `admins` account group, so the PII
columns below should show `***` while non-PII columns (`seat_gender`,
`mhtcet_score`, etc.) show real values.

In [ ]:
display(spark.sql(f'''
    SELECT candidate_name, application_id, gender, seat_gender, mhtcet_score, institute_name
    FROM {FULL_TABLE}
    LIMIT 10
'''))

## Cell 5 — Confirm Gold Is Untouched

Gold never selected these PII columns in the first place (pre-aggregated
grain: college + branch + category + gender-code, not per-candidate) — this
just re-confirms that after the fact.

In [ ]:
gold_cols = [r.col_name for r in spark.sql(f'DESCRIBE TABLE {CATALOG}.gold.mhcet_cutoffs').collect()]
leaked = set(PII_COLUMNS) & set(gold_cols)
print(f'Gold columns: {gold_cols}')
print(f'PII columns present in Gold: {leaked or "none ✅"}')